In [1]:
import pickle
import numpy as np

from pathlib import Path

from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping
)

In [2]:
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "saved_models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [3]:
with open(PROCESSED_DIR / "padding.pkl", "rb") as f:
    data = pickle.load(f)

encoder_input = data["encoder_input"]
decoder_input = data["decoder_input"]
decoder_output = data["decoder_output"]

MAX_LENGTH = data["max_length"]

print(encoder_input.shape)
print(decoder_input.shape)
print(decoder_output.shape)

(26872, 64)
(26872, 64)
(26872, 64)


In [4]:
decoder_output = np.expand_dims(decoder_output, -1)

print(decoder_output.shape)

(26872, 64, 1)


In [7]:
model = load_model(
    MODEL_DIR / "chatbot_model.keras"
)

model.summary()

d:\Development\helpdesk-ai\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 64, 64)    │    735,744 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 64)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 64, 64)    │    735,744 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     98,816 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 64, 128), │     98,816 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 64, 11496) │  1,482,984 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,456,314 (36.07 MB)

 Trainable params: 3,152,104 (12.02 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,304,210 (24.05 MB)

In [8]:
checkpoint = ModelCheckpoint(
    MODEL_DIR / "chatbot_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [10]:
history = model.fit(
    [encoder_input, decoder_input],
    decoder_output,
    batch_size=64,
    epochs=20,
    validation_split=0.1,
    callbacks=[
        checkpoint,
        early_stop
    ]
)

Epoch 1/20
378/378 ━━━━━━━━━━━━━━━━━━━━ 0s 603ms/step - accuracy: 0.1511 - loss: 4.7241
Epoch 1: val_loss improved from None to 4.32526, saving model to d:\Development\helpdesk-ai\saved_models\chatbot_model.keras

Epoch 1: finished saving model to d:\Development\helpdesk-ai\saved_models\chatbot_model.keras
378/378 ━━━━━━━━━━━━━━━━━━━━ 245s 649ms/step - accuracy: 0.1511 - loss: 4.7241 - val_accuracy: 0.2233 - val_loss: 4.3253
Epoch 2/20
378/378 ━━━━━━━━━━━━━━━━━━━━ 0s 641ms/step - accuracy: 0.3143 - loss: 3.6827
Epoch 2: val_loss improved from 4.32526 to 3.54523, saving model to d:\Development\helpdesk-ai\saved_models\chatbot_model.keras

Epoch 2: finished saving model to d:\Development\helpdesk-ai\saved_models\chatbot_model.keras
378/378 ━━━━━━━━━━━━━━━━━━━━ 259s 686ms/step - accuracy: 0.3143 - loss: 3.6827 - val_accuracy: 0.3704 - val_loss: 3.5452
Epoch 3/20
378/378 ━━━━━━━━━━━━━━━━━━━━ 0s 622ms/step - accuracy: 0.4119 - loss: 3.0252
Epoch 3: val_loss improved from 3.54523 to 3.18003,

In [11]:
with open(MODEL_DIR / "history.pkl", "wb") as f:
    pickle.dump(history.history, f)

print("✅ Training History Saved!")

✅ Training History Saved!
